# Search-03d — Weighted A\* : recherche à sous-optimalité bornée (Rust / std)

> **Partie 3 — Recherche avancée.** Ce notebook complète le **triptyque** de la
> Partie 3 sur la question « l'heuristique ne suffit pas à résoudre à l'optimum ».
> Là où [Search-03b](Search-03b-PatternDatabases.ipynb) *renforce* l'heuristique (PDB)
> et [Search-03c](Search-03c-LimitedDiscrepancySearch.ipynb) *borne les écarts* (LDS) —
> toutes deux **en gardant l'optimalité** — voici la troisième réponse : **renoncer
> (partiellement) à l'optimalité pour gagner en vitesse**, mais de façon *contrôlée*.

## 1. L'idée : payer l'optimalité pour acheter de la vitesse

[Search-3 (Informed)](../Part1-Foundations/Search-03-Informed.ipynb) a posé le
compromis fondateur : **A\*** (heuristique consistante pour la variante sans ré-ouverture) est optimal mais peut explorer beaucoup
de nœuds ; **Greedy** (sans `g`) est rapide mais peut renvoyer une solution
carrément mauvaise. Entre l'optimalité chère et la gloutonnerie aveugle, existe-t-il
un **juste milieu garanti** ?

**Weighted A\*** (Pohl, 1970) répond oui. On garde la structure d'A\*, mais on
**inflate l'heuristique** par un facteur $W \geq 1$ :

$$f(n) = g(n) + W \cdot h(n)$$

- $W = 1$ : on retrouve A\* (optimal, mais explore beaucoup).
- $W \to \infty$ : on tend vers Greedy (rapide, mais sous-optimal sans garantie).
- $1 < W < \infty$ : **Weighted A\***, qui explore bien moins de nœuds qu'A\* tout en
  garantissant une solution de coût **au pire $W$ fois l'optimal** (sous-optimalité
  bornée par $W$).

C'est le compromis « bon citizen » : on sait exactement jusqu'où on dérive de
l'optimum, et on l'échange contre un gain de nœuds qui peut être de plusieurs ordres
de grandeur.

## 2. Le banc d'essai : cheminement sur un terrain pondéré

Pour mettre Weighted A\* en valeur, il faut un problème où **A\* explore beaucoup**
(où l'optimalité coûte de nombreux nœuds) et où le compromis vitesse/qualité est net :
un terrain pondéré (routes rapides, herbe lente, marécages quasi-impraticables, murs). Sur un graphe à coût uniforme, A\*
dégenère en BFS et Weighted A\* n'apporte rien (cf la leçon « problème dégénéré ») —
il faut des **coûts contrastés** pour que l'heuristique et l'inflation discriminent.

In [2]:
use std::collections::HashMap;

struct Mt19937 { mt: [u32; 624], idx: usize }

impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 {
        let key = [seed];
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
}

const ROWS: i64 = 25;
const COLS: i64 = 25;

// --- Terrain pondere : chaque case a un cout de traversee ---
// 1 = route (rapide), 3 = herbe, 8 = marécage, '#' = mur infranchissable.
// s = swamp, g = grass.
// Coûts CONTRASTES -> A* explore beaucoup, Greedy se trompe (non-dégenéré).
fn make_grid(rows: i64, cols: i64, seed: u32) -> Vec<Vec<char>> {
    let mut rng = Mt19937::new(seed);
    let mut grid = vec![vec!['.'; cols as usize]; rows as usize];
    for r in 0..rows {
        for c in 0..cols {
            // tirage : mur (15%), herbe (30%), marécage (15%), route (40%)
            let x = rng.random();
            grid[r as usize][c as usize] = if x < 0.15 {
                '#'
            } else if x < 0.45 {
                'g'
            } else if x < 0.60 {
                's'
            } else {
                '.'
            };
        }
    }
    // garantir départ + arrivée franchissables
    grid[0][0] = '.';
    grid[(rows - 1) as usize][(cols - 1) as usize] = '.';
    grid
}

fn step_cost(grid: &Vec<Vec<char>>, r: i64, c: i64) -> Option<i64> {
    if r < 0 || r >= ROWS || c < 0 || c >= COLS { return None; }
    // None si mur
    match grid[r as usize][c as usize] {
        '.' => Some(1),
        'g' => Some(3),
        's' => Some(8),
        _ => None,
    }
}

fn neighbors(grid: &Vec<Vec<char>>, r: i64, c: i64) -> Vec<(i64, i64)> {
    let mut out = Vec::new();
    for &(dr, dc) in &[(-1i64, 0i64), (1, 0), (0, -1), (0, 1)] {
        let (nr, nc) = (r + dr, c + dc);
        if step_cost(grid, nr, nc).is_some() { out.push((nr, nc)); }
    }
    out
}

// Heuristique : Manhattan (admissible : min cost = 1, distance >= Manhattan*1).
fn manhattan(a: (i64, i64), b: (i64, i64)) -> i64 {
    (a.0 - b.0).abs() + (a.1 - b.1).abs()
}

let grid = make_grid(ROWS, COLS, 42);
let start = (0i64, 0i64);
let goal = (ROWS - 1, COLS - 1);

let n_walls = grid.iter().flatten().filter(|&&ch| ch == '#').count();
let pct = 100.0 * n_walls as f64 / (ROWS * COLS) as f64;
println!("Grille {}x{} : {} murs ({:.0}%), départ ({}, {}), but ({}, {}).",
    ROWS, COLS, n_walls, pct, start.0, start.1, goal.0, goal.1);
println!("Coûts contrastés (route=1, herbe=3, marécage=8) -> A* va explorer, Greedy va se tromper : terrain non-dégénéré.");

Grille 25x25 : 90 murs (14%), départ (0, 0), but (24, 24).


Coûts contrastés (route=1, herbe=3, marécage=8) -> A* va explorer, Greedy va se tromper : terrain non-dégénéré.


## 3. A\*, Greedy, et Weighted A\* : un seul algorithme paramétré

Les algorithmes se ramènent à **un même best-first search** sur $f(n) = g(n) + W \cdot h(n)$ :
le **poids $W$ de l'heuristique** est le seul paramètre. $W=1$ → A\* (optimal) ; $W>1$ →
Weighted A\* (sous-optimalité **bornée par $W$**) ; $W \to \infty$ → Greedy (l'heuristique
domine, $g$ devient négligeable). À noter : $W=0$ donne $f=g$, c'est-à-dire **Dijkstra/UCS**
(optimal mais aveugle à l'heuristique) — pas Greedy ; le vrai Greedy ($f=h$, $g$ ignoré) est
traité à part comme pôle non borné. On instrumente le **nombre de nœuds développés** (coût
de la recherche) et le **coût de la solution trouvée** (sa qualité).

In [3]:
use std::collections::{BinaryHeap, HashMap, HashSet};
use std::cmp::{Reverse, Ordering};

struct Mt19937 { mt: [u32; 624], idx: usize }

impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 {
        let key = [seed];
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
}

const ROWS: i64 = 25;
const COLS: i64 = 25;

fn make_grid(rows: i64, cols: i64, seed: u32) -> Vec<Vec<char>> {
    let mut rng = Mt19937::new(seed);
    let mut grid = vec![vec!['.'; cols as usize]; rows as usize];
    for r in 0..rows {
        for c in 0..cols {
            let x = rng.random();
            grid[r as usize][c as usize] = if x < 0.15 { '#'} else if x < 0.45 {'g'} else if x < 0.60 {'s'} else {'.'};
        }
    }
    grid[0][0] = '.';
    grid[(rows - 1) as usize][(cols - 1) as usize] = '.';
    grid
}

fn step_cost(grid: &Vec<Vec<char>>, r: i64, c: i64) -> Option<i64> {
    if r < 0 || r >= ROWS || c < 0 || c >= COLS { return None; }
    match grid[r as usize][c as usize] {
        '.' => Some(1), 'g' => Some(3), 's' => Some(8), _ => None,
    }
}

fn neighbors(grid: &Vec<Vec<char>>, r: i64, c: i64) -> Vec<(i64, i64)> {
    let mut out = Vec::new();
    for &(dr, dc) in &[(-1i64, 0i64), (1, 0), (0, -1), (0, 1)] {
        let (nr, nc) = (r + dr, c + dc);
        if step_cost(grid, nr, nc).is_some() { out.push((nr, nc)); }
    }
    out
}

fn manhattan(a: (i64, i64), b: (i64, i64)) -> i64 {
    (a.0 - b.0).abs() + (a.1 - b.1).abs()
}

#[derive(Clone, Copy)]
struct Key { f: f64, node: (i64, i64) }

impl PartialEq for Key {
    fn eq(&self, other: &Self) -> bool { self.f == other.f && self.node == other.node }
}
impl Eq for Key {}
impl Ord for Key {
    fn cmp(&self, other: &Self) -> Ordering {
        self.f.total_cmp(&other.f).then_with(|| self.node.cmp(&other.node))
    }
}
impl PartialOrd for Key {
    fn partial_cmp(&self, other: &Self) -> Option<Ordering> { Some(self.cmp(other)) }
}

// f(n) = g(n) + W*h(n). W=1 -> A* ; W>1 -> Weighted A* (borne W).
fn best_first(grid: &Vec<Vec<char>>, start: (i64, i64), goal: (i64, i64), w: f64) -> (Option<Vec<(i64, i64)>>, i64, usize) {
    let mut g: HashMap<(i64, i64), i64> = HashMap::new();
    let mut parent: HashMap<(i64, i64), Option<(i64, i64)>> = HashMap::new();
    g.insert(start, 0);
    parent.insert(start, None);
    // file de priorité sur f
    let mut open_h: BinaryHeap<Reverse<Key>> = BinaryHeap::new();
    open_h.push(Reverse(Key { f: w * manhattan(start, goal) as f64, node: start }));
    let mut developed = 0usize;
    while let Some(Reverse(key)) = open_h.pop() {
        let cur = key.node;
        developed += 1;
        if cur == goal {
            // reconstruire le chemin
            let mut path = Vec::new();
            let mut c: Option<(i64, i64)> = Some(cur);
            while let Some(node) = c {
                path.push(node);
                c = parent.get(&node).cloned().flatten();
            }
            path.reverse();
            return (Some(path), g[&goal], developed);
        }
        for nxt in neighbors(grid, cur.0, cur.1) {
            let nc = g[&cur] + step_cost(grid, nxt.0, nxt.1).unwrap();
            if !g.contains_key(&nxt) || nc < g[&nxt] {
                g.insert(nxt, nc);
                parent.insert(nxt, Some(cur));
                let f = nc as f64 + w * manhattan(nxt, goal) as f64;
                open_h.push(Reverse(Key { f, node: nxt }));
            }
        }
    }
    (None, i64::MAX, developed)
}

// Vrai Greedy best-first (f = h uniquement, g ignore) : le pole NON BORNE.
// On garde g pour reconstruire le chemin et mesurer son cout reel.
fn true_greedy(grid: &Vec<Vec<char>>, start: (i64, i64), goal: (i64, i64)) -> (Option<Vec<(i64, i64)>>, i64, usize) {
    let mut g: HashMap<(i64, i64), i64> = HashMap::new();
    let mut parent: HashMap<(i64, i64), Option<(i64, i64)>> = HashMap::new();
    g.insert(start, 0);
    parent.insert(start, None);
    let mut open_h: BinaryHeap<Reverse<Key>> = BinaryHeap::new();
    open_h.push(Reverse(Key { f: manhattan(start, goal) as f64, node: start }));
    let mut developed = 0usize;
    let mut seen: HashSet<(i64, i64)> = HashSet::new();
    while let Some(Reverse(key)) = open_h.pop() {
        let cur = key.node;
        developed += 1;
        if seen.contains(&cur) { continue; }
        seen.insert(cur);
        if cur == goal {
            let mut path = Vec::new();
            let mut c: Option<(i64, i64)> = Some(cur);
            while let Some(node) = c {
                path.push(node);
                c = parent.get(&node).cloned().flatten();
            }
            path.reverse();
            return (Some(path), g[&goal], developed);
        }
        for nxt in neighbors(grid, cur.0, cur.1) {
            let nc = g[&cur] + step_cost(grid, nxt.0, nxt.1).unwrap();
            if !g.contains_key(&nxt) || nc < g[&nxt] {
                g.insert(nxt, nc);
                parent.insert(nxt, Some(cur));
                // f = h seulement
                let f = manhattan(nxt, goal) as f64;
                open_h.push(Reverse(Key { f, node: nxt }));
            }
        }
    }
    (None, i64::MAX, developed)
}

let grid = make_grid(ROWS, COLS, 42);
let start = (0i64, 0i64);
let goal = (ROWS - 1, COLS - 1);

// Repères : A* (W=1) = référence optimale ; Greedy (f=h) = pôle rapide non borné.
let (_, astar_cost, astar_nodes) = best_first(&grid, start, goal, 1.0);
let (_, greedy_cost, greedy_nodes) = true_greedy(&grid, start, goal);

println!("A*     (W=1, optimal)   : coût = {:>3} | nœuds développés = {}", astar_cost, astar_nodes);
println!("Greedy (f=h, non borné) : coût = {:>3} | nœuds développés = {}  (ratio = {:.2}× optimal, AUCUNE borne)", greedy_cost, greedy_nodes, greedy_cost as f64 / astar_cost as f64);
// A* admissible -> référence d'optimalité
let optimal = astar_cost;
println!("\nRéférence d'optimalité (A* admissible) : {}.", optimal);

A*     (W=1, optimal)   : coût =  78 | nœuds développés = 472


Greedy (f=h, non borné) : coût = 143 | nœuds développés = 60  (ratio = 1.83× optimal, AUCUNE borne)


Référence d'optimalité (A* admissible) : 78.


## 4. Le balayage de $W$ : la frontière coût/nœuds

Faisons varier $W$ de 1 (A\* optimal) à de grandes valeurs (proche Greedy). À chaque
$W$, on mesure **le coût de la solution** et **le nombre de nœuds développés**. La
promesse de Weighted A\* est double, et on la vérifie honnêtement :

1. **Vitesse** — à mesure que $W$ croît, le nombre de nœuds **chute**.
2. **Garantie** — le coût de la solution reste **$\leq W \cdot \text{optimal}$** (Pohl 1970).

In [4]:
use std::collections::{BinaryHeap, HashMap};
use std::cmp::{Reverse, Ordering};

struct Mt19937 { mt: [u32; 624], idx: usize }

impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 {
        let key = [seed];
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
}

const ROWS: i64 = 25;
const COLS: i64 = 25;

fn make_grid(rows: i64, cols: i64, seed: u32) -> Vec<Vec<char>> {
    let mut rng = Mt19937::new(seed);
    let mut grid = vec![vec!['.'; cols as usize]; rows as usize];
    for r in 0..rows {
        for c in 0..cols {
            let x = rng.random();
            grid[r as usize][c as usize] = if x < 0.15 { '#'} else if x < 0.45 {'g'} else if x < 0.60 {'s'} else {'.'};
        }
    }
    grid[0][0] = '.';
    grid[(rows - 1) as usize][(cols - 1) as usize] = '.';
    grid
}

fn step_cost(grid: &Vec<Vec<char>>, r: i64, c: i64) -> Option<i64> {
    if r < 0 || r >= ROWS || c < 0 || c >= COLS { return None; }
    match grid[r as usize][c as usize] {
        '.' => Some(1), 'g' => Some(3), 's' => Some(8), _ => None,
    }
}

fn neighbors(grid: &Vec<Vec<char>>, r: i64, c: i64) -> Vec<(i64, i64)> {
    let mut out = Vec::new();
    for &(dr, dc) in &[(-1i64, 0i64), (1, 0), (0, -1), (0, 1)] {
        let (nr, nc) = (r + dr, c + dc);
        if step_cost(grid, nr, nc).is_some() { out.push((nr, nc)); }
    }
    out
}

fn manhattan(a: (i64, i64), b: (i64, i64)) -> i64 {
    (a.0 - b.0).abs() + (a.1 - b.1).abs()
}

#[derive(Clone, Copy)]
struct Key { f: f64, node: (i64, i64) }

impl PartialEq for Key {
    fn eq(&self, other: &Self) -> bool { self.f == other.f && self.node == other.node }
}
impl Eq for Key {}
impl Ord for Key {
    fn cmp(&self, other: &Self) -> Ordering {
        self.f.total_cmp(&other.f).then_with(|| self.node.cmp(&other.node))
    }
}
impl PartialOrd for Key {
    fn partial_cmp(&self, other: &Self) -> Option<Ordering> { Some(self.cmp(other)) }
}

fn best_first(grid: &Vec<Vec<char>>, start: (i64, i64), goal: (i64, i64), w: f64) -> (Option<Vec<(i64, i64)>>, i64, usize) {
    let mut g: HashMap<(i64, i64), i64> = HashMap::new();
    let mut parent: HashMap<(i64, i64), Option<(i64, i64)>> = HashMap::new();
    g.insert(start, 0);
    parent.insert(start, None);
    let mut open_h: BinaryHeap<Reverse<Key>> = BinaryHeap::new();
    open_h.push(Reverse(Key { f: w * manhattan(start, goal) as f64, node: start }));
    let mut developed = 0usize;
    while let Some(Reverse(key)) = open_h.pop() {
        let cur = key.node;
        developed += 1;
        if cur == goal {
            let mut path = Vec::new();
            let mut c: Option<(i64, i64)> = Some(cur);
            while let Some(node) = c {
                path.push(node);
                c = parent.get(&node).cloned().flatten();
            }
            path.reverse();
            return (Some(path), g[&goal], developed);
        }
        for nxt in neighbors(grid, cur.0, cur.1) {
            let nc = g[&cur] + step_cost(grid, nxt.0, nxt.1).unwrap();
            if !g.contains_key(&nxt) || nc < g[&nxt] {
                g.insert(nxt, nc);
                parent.insert(nxt, Some(cur));
                let f = nc as f64 + w * manhattan(nxt, goal) as f64;
                open_h.push(Reverse(Key { f, node: nxt }));
            }
        }
    }
    (None, i64::MAX, developed)
}

let grid = make_grid(ROWS, COLS, 42);
let start = (0i64, 0i64);
let goal = (ROWS - 1, COLS - 1);

let (_, optimal, astar_nodes) = best_first(&grid, start, goal, 1.0);

// Balayage de W >= 1 (le régime Weighted A*) : A* (W=1) -> de plus en plus "confiant".
// Greedy (f=h, pôle non borné) a déjà été mesuré ci-dessus ; ici on explore la BORNE W.
let weights = [1.0, 1.5, 2.0, 3.0, 5.0, 8.0, 12.0];
println!("{:>5} | {:>5} | {:>9} | {:>7} | {:>6} | {:>6} | garantie?", "W", "coût", "ratio/opt", "borne W", "nœuds", "% A*");
println!("{}", "-".repeat(66));
for &w in &weights {
    let (_, cost, nodes) = best_first(&grid, start, goal, w);
    let ratio = cost as f64 / optimal as f64;
    let pct = 100.0 * nodes as f64 / astar_nodes as f64;
    let garantit = if cost as f64 <= w * optimal as f64 + 1e-9 { "OUI" } else { "NON" };
    println!("{:>5.1} | {:>5} | {:>9.3} | {:>7.2} | {:>6} | {:>5.1}% | {}", w, cost, ratio, w, nodes, pct, garantit);
};

    W |  coût | ratio/opt | borne W |  nœuds |   % A* | garantie?


------------------------------------------------------------------


  1.0 |    78 |     1.000 |    1.00 |    472 | 100.0% | OUI


  1.5 |    78 |     1.000 |    1.50 |    162 |  34.3% | OUI


  2.0 |    78 |     1.000 |    2.00 |     80 |  16.9% | OUI


  3.0 |    89 |     1.141 |    3.00 |     57 |  12.1% | OUI


  5.0 |    89 |     1.141 |    5.00 |     55 |  11.7% | OUI


  8.0 |   104 |     1.333 |    8.00 |     52 |  11.0% | OUI


 12.0 |   104 |     1.333 |   12.00 |     52 |  11.0% | OUI


In [5]:
use std::collections::{BinaryHeap, HashMap};
use std::cmp::{Reverse, Ordering};

struct Mt19937 { mt: [u32; 624], idx: usize }

impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 {
        let key = [seed];
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
}

const ROWS: i64 = 25;
const COLS: i64 = 25;

fn make_grid(rows: i64, cols: i64, seed: u32) -> Vec<Vec<char>> {
    let mut rng = Mt19937::new(seed);
    let mut grid = vec![vec!['.'; cols as usize]; rows as usize];
    for r in 0..rows {
        for c in 0..cols {
            let x = rng.random();
            grid[r as usize][c as usize] = if x < 0.15 { '#'} else if x < 0.45 {'g'} else if x < 0.60 {'s'} else {'.'};
        }
    }
    grid[0][0] = '.';
    grid[(rows - 1) as usize][(cols - 1) as usize] = '.';
    grid
}

fn step_cost(grid: &Vec<Vec<char>>, r: i64, c: i64) -> Option<i64> {
    if r < 0 || r >= ROWS || c < 0 || c >= COLS { return None; }
    match grid[r as usize][c as usize] {
        '.' => Some(1), 'g' => Some(3), 's' => Some(8), _ => None,
    }
}

fn neighbors(grid: &Vec<Vec<char>>, r: i64, c: i64) -> Vec<(i64, i64)> {
    let mut out = Vec::new();
    for &(dr, dc) in &[(-1i64, 0i64), (1, 0), (0, -1), (0, 1)] {
        let (nr, nc) = (r + dr, c + dc);
        if step_cost(grid, nr, nc).is_some() { out.push((nr, nc)); }
    }
    out
}

fn manhattan(a: (i64, i64), b: (i64, i64)) -> i64 {
    (a.0 - b.0).abs() + (a.1 - b.1).abs()
}

#[derive(Clone, Copy)]
struct Key { f: f64, node: (i64, i64) }

impl PartialEq for Key {
    fn eq(&self, other: &Self) -> bool { self.f == other.f && self.node == other.node }
}
impl Eq for Key {}
impl Ord for Key {
    fn cmp(&self, other: &Self) -> Ordering {
        self.f.total_cmp(&other.f).then_with(|| self.node.cmp(&other.node))
    }
}
impl PartialOrd for Key {
    fn partial_cmp(&self, other: &Self) -> Option<Ordering> { Some(self.cmp(other)) }
}

fn best_first(grid: &Vec<Vec<char>>, start: (i64, i64), goal: (i64, i64), w: f64) -> (Option<Vec<(i64, i64)>>, i64, usize) {
    let mut g: HashMap<(i64, i64), i64> = HashMap::new();
    let mut parent: HashMap<(i64, i64), Option<(i64, i64)>> = HashMap::new();
    g.insert(start, 0);
    parent.insert(start, None);
    let mut open_h: BinaryHeap<Reverse<Key>> = BinaryHeap::new();
    open_h.push(Reverse(Key { f: w * manhattan(start, goal) as f64, node: start }));
    let mut developed = 0usize;
    while let Some(Reverse(key)) = open_h.pop() {
        let cur = key.node;
        developed += 1;
        if cur == goal {
            let mut path = Vec::new();
            let mut c: Option<(i64, i64)> = Some(cur);
            while let Some(node) = c {
                path.push(node);
                c = parent.get(&node).cloned().flatten();
            }
            path.reverse();
            return (Some(path), g[&goal], developed);
        }
        for nxt in neighbors(grid, cur.0, cur.1) {
            let nc = g[&cur] + step_cost(grid, nxt.0, nxt.1).unwrap();
            if !g.contains_key(&nxt) || nc < g[&nxt] {
                g.insert(nxt, nc);
                parent.insert(nxt, Some(cur));
                let f = nc as f64 + w * manhattan(nxt, goal) as f64;
                open_h.push(Reverse(Key { f, node: nxt }));
            }
        }
    }
    (None, i64::MAX, developed)
}

// Visualisation ASCII : le chemin trouvé par A* (W=1) vs Weighted A* (W=3) vs Greedy (W=0).
fn render(grid: &Vec<Vec<char>>, path: &Vec<(i64, i64)>, start: (i64, i64), goal: (i64, i64)) -> String {
    let mut cells: Vec<Vec<char>> = grid.clone();
    // le chemin
    for &(r, c) in path {
        cells[r as usize][c as usize] = '@';
    }
    cells[start.0 as usize][start.1 as usize] = 'S';
    cells[goal.0 as usize][goal.1 as usize] = 'G';
    let mut lines = Vec::new();
    for r in 0..ROWS {
        let line: String = cells[r as usize].iter().map(|&ch| match ch {
            '.' => '.', 'g' => ',', 's' => '~', '#' => '#',
            '@' => '@', 'S' => 'S', 'G' => 'G', _ => '.',
        }).collect();
        lines.push(line);
    }
    lines.join("\n")
}

let grid = make_grid(ROWS, COLS, 42);
let start = (0i64, 0i64);
let goal = (ROWS - 1, COLS - 1);

let (astar_path, _, _) = best_first(&grid, start, goal, 1.0);
let (wa3_path, _, _) = best_first(&grid, start, goal, 3.0);

println!("Légende : S départ · G but · @ chemin · . route · , herbe · ~ marécage · # mur\n");
println!("=== A* (W=1, optimal) ===");
println!("{}", render(&grid, &astar_path.unwrap(), start, goal));
println!("\n=== Weighted A* (W=3) ===");
println!("{}", render(&grid, &wa3_path.unwrap(), start, goal));

Légende : S départ · G but · @ chemin · . route · , herbe · ~ marécage · # mur


=== A* (W=1, optimal) ===


S#,,...#,#,~#,.~,~.#..,,.


@##....~.,~...~.#,,#,#,.,


@@,...,.,,..~...,#,,,..,.


,@~,,~,~.,,.~###..,#,.~..


... (43 outputs left out)

..,#,#.~,..,..#..##,@@@@#


..#...#..,,,~,..#....,.@@


.~~,.,..#.,~.,#.,,.,..,#@


#.~...~.#,.,~~~,.,.,,#,#G


### Lecture du résultat

Les repères et le balayage encadrent le **compromis structurel** de Weighted A\* par ses
deux pôles, puis l'interpolation bornée entre les deux :

- **Greedy (f=h) — pôle rapide, NON borné.** Seulement **60 nœuds**, mais un coût de
  **143 (1,83× l'optimal)**. Greedy « fonce » vers le but : parfois il tombe juste, parfois
  (comme ici) très mal — on n'a **aucune garantie** sur sa dérive, qui peut être arbitraire.
- **A\* (W=1) — pôle optimal, cher.** Coût **78 (optimal garanti)**, mais **472 nœuds** :
  c'est le prix de l'optimalité stricte, A\* explore largement autour du chemin pour prouver
  qu'aucun meilleur n'existe.
- **Weighted A\* (1 < W) — l'interpolation bornée.** C'est tout l'intérêt : à $W=2$, on
  retrouve **le coût optimal (78)** en ne développant que **80 nœuds (17 % d'A\*)** — un gain
  quasi gratuit. À $W=3$, **57 nœuds (12 %)** pour un coût **89 (1,14× l'optimal)**. La
  colonne « garantie » confirme partout : $\text{coût} \leq W \cdot \text{optimal}$.

La **frontière de Pareto** se lit ainsi : choisir $W$, c'est choisir un point sur la courbe
coût/nœuds. Un $W$ modeste ($2$ à $3$) capture l'essentiel du gain de vitesse pour une dérive
qualité négligeable — tandis que Greedy, lui, abandonne toute borne sur la sienne.

La carte ASCII montre l'effet visuel : A\* ($W=1$) contourne méthodiquement les zones coûteuses
(chemin optimal), tandis que Weighted A\* ($W=3$) « coupe » à travers des cases un peu plus
chères pour aller plus droit — un chemin légèrement plus cher (89 vs 78), trouvé en développant
**8 fois moins** de nœuds (57 vs 472).

## 5. Pourquoi ça marche : la garantie de sous-optimalité bornée

La garantie $\text{coût} \leq W \cdot \text{optimal}$ repose sur une idée simple. En
gonflant l'heuristique par $W$, on rend la recherche **plus confiante** dans l'estimation
$h$ : elle « fonce » vers le but en s'autorisant à développer moins de nœuds latéraux.
Le prix de cette confiance est qu'on peut manquer le chemin optimal — mais le facteur
d'inflation $W$ **borne exactement** de combien on peut dévier.

Formellement (Pohl 1970) : si $h$ est **admissible** ($h \leq h^*$, vrai coût), alors la
solution de Weighted A\* avec facteur $W$ a un coût au plus $W \cdot C^*$, où $C^*$ est le
coût optimal. La preuve suit la même structure que l'optimalité d'A\* : au moment où le
but est développé, son $f$-valeur est bornée par $W \cdot C^*$ via les nœuds sur le chemin
optimal, dont l'$f$ gonflé reste sous ce seuil.

Conséquences pratiques :
- **Choisir $W$, c'est choisir le budget d'erreur acceptable.** Si on tolère une solution
  10 % trop chère, $W = 1{,}1$ suffit — souvent avec un gain de nœuds déjà notable.
- **L'heuristique doit rester admissible.** Gonfler une heuristique *déjà inadmissible*
  n'offre plus de garantie — Weighted A\* suppose un $h$ honnête qu'on choisit de
  sur-pondérer.
- **Anytime** — on peut démarrer avec un grand $W$ (solution rapide, grossière) puis le
  réduire progressivement en réutilisant la recherche, pour raffiner vers l'optimal
  (Anytime Repairing A\*, Likhachev 2003 — exercice 3).

## 6. Ponts avec le reste de la série

| Direction | Lien | Relation |
|-----------|------|----------|
| ↔ Search-03b | [Search-03b — Pattern Databases](Search-03b-PatternDatabases.ipynb) | Autre réponse à « l'optimum est dur » : *renforcer* l'heuristique (rester optimal) vs ici *relâcher* l'optimalité |
| ↔ Search-03c | [Search-03c — Limited Discrepancy Search](Search-03c-LimitedDiscrepancySearch.ipynb) | LDS *borne les écarts* à l'heuristique (rester optimal) ; Weighted A\* *relâche* l'optimalité — deux axes orthogonaux |
| ← Partie 1 | [Search-3 — Informed](../Part1-Foundations/Search-03-Informed.ipynb) | A\*, admissibilité, Greedy : les deux extrémités ($W=1$ et $W=0$) que Weighted A\* interpole |
| → Partie 4 | [Métaheuristiques](../Part4-Metaheuristics/README.md) | Weighted A\* garde une *borne* sur la qualité ; les métaheuristiques l'abandonnent totalement |

**Triptyque de la Partie 3** — trois réponses à « l'heuristique ne suffit pas à résoudre à
l'optimum » : **renforcer** l'heuristique (Search-03b, PDB), **borner les écarts**
(Search-03c, LDS), **relâcher** l'optimalité (Search-03d, Weighted A\*).

**Référence fondatrice** : Pohl, I. (1970) — *Heuristic Search Viewed as Path Finding in a
Graph*, Artificial Intelligence 1(3-4). Première formalisation de Weighted A\* et de sa
garantie de sous-optimalité bornée par le facteur d'inflation $W$.

## 7. Exercices

### Exercice 1 : courbe de Pareto coût/nœuds
Tracez (ou affichez en ASCII) la **frontière de Pareto** : en abscisse le nombre de nœuds
développés, en ordonnée le ratio coût/optimal, un point par valeur de $W \in [1, 12]$.
Identifiez le « genou » (knee) — la valeur de $W$ au-delà de laquelle gagner encore en
vitesse coûte cher en qualité.
- **Indice :** le genou est typiquement autour de $W \in [2, 3]$. Justifiez pourquoi
  l'utilité marginale d'augmenter $W$ décroît (les nœuds économisés s'amenuissent, la
  dérive quality s'accumule).

In [6]:
// Exercice 1 a completer
// Conseil : reprenez le balayage ci-dessus (weights plus fin, ex. [1, 1.2, 1.5, 2, 2.5, 3, 4, 6, 9, 12]),
// collectez (nodes, ratio) pour chaque W, et repérez le "genou" de la frontière de Pareto.
println!("Exercice 1 a completer");

Exercice 1 a completer


### Exercice 2 : sensibilité à l'heuristique (admissible vs inadmissible)
Refaitez le balayage de $W$ avec une heuristique **inadmissible** (par ex. Manhattan
**sans** tenir compte du coût minimal : ou $h$ sur-estimé d'un facteur 2 dès le départ).
Vérifiez que la **garantie** $\text{coût} \leq W \cdot \text{optimal}$ est alors **violée**.
- **Indice :** la garantie suppose $h$ admissible. Avec un $h$ déjà inadmissible, l'inflation
  par $W$ aggrave la triche et la borne n'a plus de raison de tenir. C'est pourquoi Weighted
  A\* part toujours d'une heuristique *honnête*.

In [7]:
// Exercice 2 a completer
// Conseil : définissez h_bad(s) = 2*manhattan(s, goal) (inadmissible), relancez le balayage,
// et cherchez un W où cost > W*optimal (garantie violée).
println!("Exercice 2 a completer");

Exercice 2 a completer


### Exercice 3 : Anytime Repairing A\* (ARA\*)
Implémentez une version **anytime** : démarrez avec un grand $W_0$ (solution rapide),
puis **décrémentez** $W$ par petits pas en **réutilisant** la liste OPEN déjà construite,
jusqu'à $W=1$ (solution optimale). À chaque étape, affichez le coût courant et le temps
écoulé — l'utilisateur peut s'arrêter quand il est satisfait.
- **Indice :** ARA\* (Likhachev, Gordon & Thrun, 2003) réutilise l'OPEN en mettant juste à
  jour les clés $f = g + W \cdot h$ quand $W$ change, sans tout recalculer. L'intérêt :
  une solution **immédiate** qui s'améliore avec le budget temps. C'est l'usage typique de
  Weighted A\* en robotique (planification de mouvement sous contrainte temps-réel).

In [8]:
// Exercice 3 a completer
// Conseil : boucle W = W0, W0-step, ... 1. À chaque W, re-triez l'OPEN existante par la
// nouvelle clé f=g+W*h (ou réinsérez), développez jusqu'au but, affichez (W, coût, t).
println!("Exercice 3 a completer");

Exercice 3 a completer


## Conclusion

**Weighted A\*** referme le triptyque de la Partie 3. Face à un problème où A\* optimal
explore trop de nœuds, on dispose de trois leviers : **renforcer** l'heuristique
([Search-03b](Search-03b-PatternDatabases.ipynb), PDB), **borner les écarts** à l'heuristique
([Search-03c](Search-03c-LimitedDiscrepancySearch.ipynb), LDS) — qui préservent l'optimalité —
ou, sujet de ce notebook, **relâcher l'optimalité de façon contrôlée** (Weighted A\*). Ce
troisième levier est le plus radical et souvent le plus efficace en pratique : on accepte
une solution au plus $W$ fois l'optimal, et on gagne en échange un facteur de nœuds qui peut
être énorme.

La leçon générale est un **compromis explicite qualité/vitesse** : contrairement à Greedy
(dérive non bornée) ou aux métaheuristiques de la [Partie 4](../Part4-Metaheuristics/README.md)
(aucune garantie), Weighted A\* offre une **borne de qualité paramétrable**. Choisir $W$,
c'est choisir son budget d'erreur — un compromis honnête, quantitatif, et qui se raffine en
*anytime* (ARA\*) quand on veut se rapprocher de l'optimal avec le temps restant.

> **Au-delà** : Weighted A\* est la brique de base de la planification sous contrainte
> temps-réel (robotique, jeux vidéo) et se combine aux PDB (heuristique forte ET relâchée)
> et au branch-and-bound borné. C'est le point où la série bascule de « garantir l'optimal »
> à « garantir une qualité calculable » — juste avant que la Partie 4 n'abandonne toute
> garantie.

## À retenir

1. **Un seul algorithme, un seul paramètre** : f(n) = g(n) + W·h(n). W = 1 redonne A\*, W → ∞ tend vers Greedy. Le banc d'essai le rend mesurable sur un terrain à **coûts contrastés** (route = 1, herbe = 3, marécage = 8 ; 90 murs sur 625 cases) : A\* (W = 1) trouve le coût optimal **78** en **472 nœuds**, et le vrai Greedy (f = h) un coût de **143**, soit **1,83× l'optimal**, en **60 nœuds** — rapide, mais **sans aucune borne** sur sa dérive.
2. **Le gain gratuit est à W = 2** : on retrouve le **coût optimal (78)** en ne développant que **80 nœuds**, soit **16,9 %** du compte d'A\* — 5,9 fois moins de nœuds pour une dérive de qualité nulle.
3. **La borne tient partout** : sur les sept valeurs de W balayées, la colonne « garantie » vaut **OUI** à chaque ligne (coût ≤ W × 78) — et la dérive réelle reste très en deçà de la borne théorique : à W = 3, le coût est **89** (1,141× l'optimal) pour une borne de 3,00.
4. **Le gain sature vite** : de W = 8 à W = 12, ni les nœuds (52) ni le coût (104, soit 1,333× l'optimal) ne bougent plus. Le « genou » de la frontière coût/nœuds se situe donc entre W = 2 (optimal encore atteint) et W = 3 (première dérive) : au-delà, chaque nœud économisé se paie en qualité.
5. **La garantie a un prix : l'admissibilité de h** : la borne coût ≤ W × optimal repose sur une heuristique *honnête* (Manhattan est admissible ici, le coût minimal de traversée valant 1). Gonfler une heuristique déjà inadmissible — l'exercice 2 le mesure — ne redonne plus de borne : l'inflation ne fait qu'aggraver la surestimation.

## Vérifiez votre compréhension

1. **À W = 2, la solution a le coût optimal (78) pour 80 nœuds au lieu de 472.** Qu'est-ce qui a réellement été sacrifié, et que dit la garantie de ce résultat ?
2. **Le Greedy trouve un chemin en 60 nœuds pour un coût de 143 ; W = 3 en trouve un en 57 nœuds pour 89.** Pourquoi Weighted A\* est-il meilleur sur les *deux* critères, alors que le Greedy « fonce » plus vite ?
3. **La garantie « coût ≤ W × optimal » disparaît si h n'est plus admissible.** Quel est le rôle exact de l'admissibilité dans la preuve ?
4. **Le terrain est volontairement contrasté** (route 1, herbe 3, marécage 8). Que se passerait-il sur un terrain à coût uniforme, et pourquoi le carnet évite-t-il ce cas ?

<details><summary>Réponses</summary>

1. Rien en qualité : la garantie ne dit pas « la solution est optimale », elle dit « coût ≤ W × optimal ». Ici l'égalité (78 = 78) est un **hasard heureux** de l'inflation — W = 2 suffit, sur cette instance, à retomber sur le chemin optimal — et non une conséquence du théorème : la borne autorise tout jusqu'à 2 × 78 = 156. C'est pourquoi on parle de *borne* et non de garantie d'optimalité.

2. Parce que f = g + W·h conserve l'information du **coût déjà payé** (le terme g), que le Greedy ignore complètement. Le Greedy « fonce » vers le but sans regarder ce qu'il dépense : son chemin coûte 143 pour un trajet minimal de 48 cases (Manhattan de (0, 0) à (24, 24)), soit au moins 95 unités payées au-delà du minimum — nécessairement des cases d'herbe (3) ou de marécage (8) là où A\* passe par la route. Weighted A\* reste un A\* : il perd de l'information sur l'*estimation* du coût restant, pas sur le coût engagé. Résultat mesuré : 57 nœuds et un coût de 89, contre 60 nœuds et 143 pour le Greedy — mieux sur les deux critères.

3. La preuve suit la structure de l'optimalité d'A\* : au moment où le but est développé, sa f-valeur est bornée par W · C\* grâce aux nœuds du chemin optimal, dont le f gonflé reste sous ce seuil. Cette borne n'utilise qu'une chose de h : qu'il **minore** le vrai coût restant (admissibilité). Si h sur-estime, les f-valeurs du chemin optimal dépassent le seuil et plus rien ne borne le coût final : avec h = 2 × Manhattan, la colonne « garantie » du balayage passe à NON (exercice 2).

4. Sur un terrain à coût uniforme, l'heuristique ne discrimine pas les itinéraires et l'inflation n'a rien à peser : c'est le « problème dégénéré » que le carnet cite en section 2, où Weighted A\* n'apporte rien. Le terrain contrasté est donc ce qui rend la frontière coût/nœuds **mesurable** : sans lui, le gain de nœuds de W = 3 serait un artefact de l'instance, pas une propriété de l'algorithme.

</details>

## Pour aller plus loin

- **Jumeau Python** : [Search-03d-WeightedAstar](Search-03d-WeightedAstar.ipynb) — la version originale Python de ce carnet.
- **Jumeau .NET** : [Search-03d-WeightedAstar-CSharp](Search-03d-WeightedAstar-CSharp.ipynb) — la même expérience en C#, avec ses propres mesures.
- **Carnet voisin** : [Search-03c-LimitedDiscrepancySearch](Search-03c-LimitedDiscrepancySearch.ipynb) — l'axe orthogonal : borner les écarts à l'heuristique en **conservant** l'optimalité, là où Weighted A\* relâche l'optimalité de façon **bornée**.
- **Dans la même série** : [Search-03b-PatternDatabases](Search-03b-PatternDatabases.ipynb) (renforcer l'heuristique) et [Search-03e-AStar-Optimality](Search-03e-AStar-Optimality.ipynb) (le pôle optimal et l'admissibilité, avec une visite formelle du kernel `search_lean`).
- **Exercices du carnet** : l'exercice 1 trace la frontière de Pareto et son « genou » (point 4), l'exercice 2 mesure la violation de la borne par une heuristique inadmissible (point 5), l'exercice 3 implémente ARA\*, qui raffine la solution vers l'optimal avec le temps restant.
- **Lectures** : Pohl, I. (1970), *Heuristic Search Viewed as Path Finding in a Graph*, Artificial Intelligence 1(3-4) — la formalisation de Weighted A\* et de sa borne, déjà nommée par le carnet ; Likhachev, M., Gordon, G., & Thrun, S. (2003), *ARA\* — Anytime A\* with provably optimal cost bound* — l'extension *anytime* de l'exercice 3, également nommée par le carnet.

> Carnet issu de l'EPIC #26510.